### 使用 GPT-4o 将音频文件语音翻译成不同语言

你是否曾想把一期播客翻译成你的母语？翻译和配音音频内容可以让全球受众更容易获取这些内容。借助 GPT-4o 新的音频输入、音频输出模态，这一过程变得前所未有地简单。

本指南将带你使用 OpenAI 的 GPT-4o 音频模态 API，把一个英语音频文件翻译成印地语。

GPT-4o 简化了音频内容的配音流程。以前，你必须先把音频转换为文本，再把文本翻译成目标语言，然后再转换回音频。现在，借助 GPT-4o 的语音到语音能力，你可以通过音频输入和输出一步完成。  

关于本 Cookbook 中**语言（Language）**和书写**文字（Script）**的用词说明。这两个词通常可以互换使用，但鉴于手头的任务，理解它们的区别很重要。  
 
**- 语言**指口头或书面的交流体系。例如，印地语和马拉地语是不同的语言，但都使用天城文。同样，英语和法语是不同的语言，但都用拉丁字母书写。  
   
**- 文字**指用于书写语言的一套字符或符号。例如，塞尔维亚语传统上用西里尔字母书写，也可以用拉丁字母书写。


GPT-4o 的音频输入、音频输出模态让你只需一次 API 调用，就能更轻松地将音频从一种语言配音成另一种语言。  

**1. 转写**：使用 GPT-4o 将源音频文件转写为源语言文字。如果你已有源音频内容的转写文本，可以跳过这一可选步骤。  
 
**2. 配音**：将音频文件从源语言直接配音为目标语言。  
   
**3. 获取翻译基准分数**：使用 BLEU 或 ROUGE。   
 
**4. 解读并提升**分数：根据需要调整第 1-3 步中的提示参数。   


在开始之前，请确保已将 OpenAI API 密钥配置为环境变量，并按照下面代码单元格中的说明安装了必要的包。 

### 第 1 步：使用 GPT-4o 将音频转写为源语言文字 

首先，我们创建一个函数，通过 chat completions API 端点将音频文件发送到 OpenAI 的 GPT-4o API 进行处理。

函数 `process_audio_with_gpt_4o` 接受三个输入：

1. 一个 base64 编码的音频文件（base64_encoded_audio），它将被发送到 GPT-4o 模型。
2. 期望的输出模态（例如文本，或文本加音频）。 
3. 一个系统提示，指示模型如何处理输入。

该函数向 OpenAI 的 chat/completions 端点发送 API 请求。请求头中包含用于授权的 API 密钥。数据负载包含模型类型（`gpt-4o-audio-preview`）、所选的输出模态以及音频详情，例如声音类型和格式（本例中为 "alloy" 和 "wav"）。它还将系统提示和 base64 编码的音频文件作为 "user" 消息的一部分包含在内。如果 API 请求成功（HTTP 状态码 200），响应会以 JSON 形式返回。如果发生错误（非 200 状态码），则会打印错误代码和消息。

该函数通过 OpenAI 的 GPT-4o API 实现音频处理，可根据提供的输入执行配音、转写或翻译等任务。

In [1]:
# Make sure requests package is installed  
import requests 
import os
import json

# Load the API key from the environment variable
api_key = os.getenv("OPENAI_API_KEY")


def process_audio_with_gpt_4o(base64_encoded_audio, output_modalities, system_prompt):
    # Chat Completions API end point 
    url = "https://api.openai.com/v1/chat/completions"

    # Set the headers
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }

    # Construct the request data
    data = {
        "model": "gpt-4o-audio-preview",
        "modalities": output_modalities,
        "audio": {
            "voice": "alloy",
            "format": "wav"
        },
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_audio",
                        "input_audio": {
                            "data": base64_encoded_audio,
                            "format": "wav"
                        }
                    }
                ]
            }
        ]
    }
    
    request_response = requests.post(url, headers=headers, data=json.dumps(data))
    if request_response.status_code == 200:
        return request_response.json()
    else:  
        print(f"Error {request_response.status_code}: {request_response.text}")
        return
    

使用函数 `process_audio_with_gpt_4o`，我们首先获取源音频的英语转写文本。如果你已有源语言的转写文本，可以跳过这一步。 

在这一步中，我们： 
1. 读取 WAV 文件并将其转换为 base64 编码。
2. 将输出模态设置为 ["text"]，因为我们只需要文本转写。
3. 提供系统提示，指示模型专注于转写语音，忽略掌声等背景噪音。
4. 调用 process_audio_with_gpt_4o 函数处理音频并返回转写文本。

In [2]:
import base64
audio_wav_path = "./sounds/keynote_recap.wav"

# Read the WAV file and encode it to base64
with open(audio_wav_path, "rb") as audio_file:
    audio_bytes = audio_file.read()
    english_audio_base64 = base64.b64encode(audio_bytes).decode('utf-8')

modalities = ["text"]
prompt = "The user will provide an audio file in English. Transcribe the audio to English text, word for word. Only provide the language transcription, do not include background noises such as applause. "

response_json = process_audio_with_gpt_4o(english_audio_base64, modalities, prompt)

english_transcript = response_json['choices'][0]['message']['content']

print(english_transcript)

Hello and welcome to our first ever OpenAI DevDay. Today, we are launching a new model, GPT-4 Turbo. GPT-4 Turbo supports up to 128,000 tokens of context. We have a new feature called JSON mode, which ensures that the model will respond with valid JSON. You can now call many functions at once. And it will do better at following instructions in general. You want these models to be able to access better knowledge about the world, so do we. So we're launching retrieval in the platform. You can bring knowledge from outside documents or databases into whatever you're building. GPT-4 Turbo has knowledge about the world up to April of 2023, and we will continue to improve that over time. DALL-E 3, GPT-4 Turbo with Vision, and the new text-to-speech model are all going into the API today. Today, we're launching a new program called Custom Models. With Custom Models, our researchers will work closely with a company to help them make a great custom model, especially for them and their use case u

在第 3 步对音频的印地语配音进行基准评估时，这份英语转写文本将作为我们的参考标准。 

### 第 2 步：使用 GPT-4o 将音频从源语言配音为目标语言

借助 GPT-4o，我们只需一次 API 调用，就能把音频文件直接从英语配音为印地语，并获得该音频的印地语转写文本。为此，我们将输出模态设置为 `["text", "audio"] `

In [3]:
glossary_of_terms_to_keep_in_original_language = "Turbo, OpenAI, token, GPT, Dall-e, Python"

modalities = ["text", "audio"]
prompt = f"The user will provide an audio file in English. Dub the complete audio, word for word in Hindi. Keep certain words in English for which a direct translation in Hindi does not exist such as  ${glossary_of_terms_to_keep_in_original_language}."

response_json = process_audio_with_gpt_4o(english_audio_base64, modalities, prompt)

message = response_json['choices'][0]['message']

在下面的代码片段中，我们将从 GPT-4o 的响应中同时获取印地语转写文本和配音后的音频。以前，这需要一个多步骤流程，涉及多次 API 调用：先转写，再翻译，最后生成目标语言的音频。现在借助 GPT-4o，我们只需一次 API 调用即可完成。

In [4]:
# Make sure pydub is installed 
from pydub import AudioSegment
from pydub.playback import play
from io import BytesIO

# Get the transcript from the model. This will vary depending on the modality you are using. 
hindi_transcript = message['audio']['transcript']

print(hindi_transcript)

# Get the audio content from the response 
hindi_audio_data_base64 = message['audio']['data']


स्वागत है हमारे पहले OpenAI DevDay में।

आज हम एक नए मॉडल का लॉन्च कर रहे हैं, GPT-4 Turbo। GPT-4 Turbo अब 1,28,000 टोकens के कॉन्टेक्स्ट को सपोर्ट करता है। हमारे पास एक नया फीचर है जिसे JSON मोड कहा जाता है, जो सुनिश्चित करता है कि मॉडल वैध JSON के साथ प्रतिक्रिया करेगा। अब आप कई फंक्शन्स को एक साथ कॉल कर सकते हैं। और ये सामान्य रूप से इंस्ट्रक्शंस का पालन करने में बेहतर करेगा। आप चाहते हैं कि ये मॉडल दुनिया के बारे में बेहतर जानकारी तक पहुंच सकें, हम भी। इसलिए हम प्लैटफॉर्म में Retrieval लॉन्च कर रहे हैं। आप बाहरी दस्तावेज़ या डेटाबेस से जो भी आप बना रहे हैं, उसमें ज्ञान ला सकते हैं। GPT-4 Turbo को अप्रैल 2023 तक की दुनिया की जानकारी है, और हम इसे समय के साथ और बेहतर बनाना जारी रखेंगे। DALL·E 3, GPT-4 Turbo with vision, और नया Text-to-Speech मॉडल सभी को आज उपलब्ध कर रहे हैं एपीआई में। आज हम एक नए प्रोग्राम का लॉन्च कर रहे हैं जिसे Custom Models कहा जाता है। Custom Models के साथ, हमारे शोधकर्ता एक कंपनी के साथ निकटता से काम करेंगे ताकि वे एक महान Custom Model बना सकें, विशेष रूप से उन

转写文本是印地语和英语的混合，分别以各自的文字表示：印地语使用天城文，英语使用拉丁字母。这种方式可以确保语音听起来更自然，两种语言的单词发音都正确。我们将使用 `pydub` 模块播放音频，如下面的代码所示。 

In [5]:
# Play the audio 
audio_data_bytes = base64.b64decode(hindi_audio_data_base64)
audio_segment = AudioSegment.from_file(BytesIO(audio_data_bytes), format="wav")

play(audio_segment)

### 第 3 步：获取翻译基准分数（例如 BLEU 或 ROUGE） 

我们可以使用 BLEU 和 ROUGE 等评估指标，将翻译文本与参考译文进行比较，从而评估翻译文本的质量。 

**BLEU（Bilingual Evaluation Understudy）**：衡量候选译文与参考译文之间 n-gram 的重叠程度。分数范围为 0 到 100，分数越高表示质量越好。

**ROUGE（Recall-Oriented Understudy for Gisting Evaluation）**：常用于摘要评估。衡量候选文本与参考文本之间 n-gram 的重叠程度以及最长公共子序列。

理想情况下，要进行准确的评估，需要原文的参考译文（人工翻译的版本）。然而，开发这样的评估可能颇具挑战，因为这需要精通两种语言的双语人员投入时间和精力。

另一种方法是使用 GPT-4o 将目标语言的输出音频文件转写回原始语言，以评估翻译质量。 

In [6]:
# Translate the audio output file generated by the model back into English and compare with the reference text 
modalities = ["text"]
prompt = "The user will provide an audio file in Hindi. Transcribe the audio to English text word for word. Only provide the language transcription, do not include background noises such as applause. "

response_json = process_audio_with_gpt_4o(hindi_audio_data_base64, modalities, prompt)

re_translated_english_text = response_json['choices'][0]['message']['content']

print(re_translated_english_text)

Welcome to our first OpenAI Dev Day. Today we are launching a new model, GPT-4 Turbo. GPT-4 Turbo now supports a context of 128,000 tokens. We have a new feature called JSON mode where the model will respond via JSON. Now you can call multiple functions simultaneously, and it will generally follow instructions better. You want this model to access external knowledge databases or documents to bring knowledge into what you are building. GPT-4 Turbo has knowledge of the world up to April 2023, and we'll continue to improve it over time.

DALL·E 3, GPT-4 Turbo with vision, and the new text-to-speech model are all being made available today in the API. Today, we are launching a new program called custom models. Custom models will work closely to make great custom models specifically for you and your use case. Utilizing our tools, we are doubling the rate limits for all existing GPT-4 customers to tokens per minute. You'll be able to directly request rate limit and quota changes in your API 

将印地语音频转写回英语文字后，我们就可以将其与原始英语转写文本进行比较，计算评估指标。

In [7]:
# Make sure scarebleu package is installed 
import sacrebleu
# Make sure rouge-score package is installed 
from rouge_score import rouge_scorer 

# We'll use the original English transcription as the reference text 
reference_text = english_transcript

candidate_text = re_translated_english_text

# BLEU Score Evaluation
bleu = sacrebleu.corpus_bleu([candidate_text], [[reference_text]])
print(f"BLEU Score: {bleu.score}")

# ROUGE Score Evaluation
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference_text, candidate_text)
print(f"ROUGE-1 Score: {scores['rouge1'].fmeasure}")
print(f"ROUGE-L Score: {scores['rougeL'].fmeasure}")

BLEU Score: 35.27656890256424
ROUGE-1 Score: 0.8148148148148149
ROUGE-L Score: 0.6934156378600822


### 第 4 步：根据需要调整第 1-3 步中的提示参数，以解读并提升分数

在本例中，BLEU 和 ROUGE 分数都表明语音翻译的质量介于非常好和优秀之间。

**解读 BLEU 分数：**虽然没有公认的统一标准，但一些解读认为：

0 到 10：翻译质量差；存在明显错误，缺乏流畅性。

10 到 20：质量低；部分内容可以理解，但包含许多错误。

20 到 30：质量一般；能传达大意，但缺乏准确性和流畅性。

30 到 40：质量良好；可以理解且相对准确，只有少量错误。

40 到 50：质量非常好；准确流畅，错误极少。

50 及以上：质量优秀；与人工翻译非常接近。

**解读 ROUGE 分数：**怎样的 ROUGE 分数算“好”，可能因任务、数据集和领域而异。以下准则表示结果良好：

ROUGE-1（unigram 重叠）：对于生成式摘要任务，0.5 到 0.6 之间的分数通常被认为是好的。

ROUGE-L（最长公共子序列）：0.4 到 0.5 左右的分数通常被认为是好的，反映了模型捕捉参考文本结构的能力。

如果你的翻译分数不尽如人意，可以考虑以下问题：

#### 1. 源音频是否被准确转写？ 
如果转写文本中包含错误，例如混淆了发音相似的单词，你可以在第 1 步的系统提示中提供这类术语的词汇表。这有助于模型避免误解，并确保特定术语被准确转写。

#### 2. 源音频是否没有语法错误？ 
如果源音频包含语法错误，可以考虑使用 GPT 模型进行后处理，通过去除语法错误并添加恰当的标点来完善转写文本。之后，你可以不使用 GPT-4o 的音频输入、音频输出模态，而是将修正后的转写文本与 GPT-4o 的文本输入、音频输出模态结合使用，生成目标语言的音频。

#### 3. 是否有些词保留原始语言更合理？  
某些术语或概念在目标语言中可能没有合适的译法，或者保留原文形式更容易理解。请重新检查你的 `glossary_of_terms_to_keep_in_original_language`，加入此类术语，以保持清晰度和上下文。

### 总结

总之，本 cookbook 提供了一个清晰的、分步骤的音频翻译和配音流程，让内容更容易被全球受众获取。借助 GPT-4o 的音频输入和输出能力，将音频文件从一种语言翻译并配音成另一种语言变得简单得多。我们的示例重点介绍了将音频文件从英语翻译成印地语。

该流程可以分解为以下步骤：

**1. 转写：**使用 GPT-4o 的文本模态，将源语言音频转写为源语言文字。   

**2. 配音：**使用 GPT-4o 的音频模态，将音频文件直接配音为目标语言。  

**3. 评估翻译质量：**与参考文本进行比较，使用 BLEU 或 ROUGE 分数评估翻译的准确性。

**4. 优化流程：**如有需要，调整提示参数以改进转写和配音结果。  

本指南还强调了“语言”和“文字”之间的关键区别——这两个术语经常被混淆，但在翻译工作中至关重要。语言指口头或书面的交流体系，而文字是用于书写语言的一套字符。理解这一区别对于有效的翻译和配音至关重要。

按照本 cookbook 中的技术，你可以将各种内容——从播客、培训视频到完整的电影——翻译并配音成多种语言。这种方法适用于娱乐、教育、商业和全球沟通等各个行业，帮助创作者将影响力扩展到不同语言的受众。